In [1]:
# ========================================================
# 1. INSTALL LIBRARY & OLLAMA
# ========================================================
!apt-get update && apt-get install zstd -y
!curl -fsSL https://ollama.com/install.sh | sh
!pip install langchain langchain_community langchain-ollama langchainhub sentence-transformers langchain-huggingface bert-score

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [2]:
import os
import subprocess
import time
import json
import torch
import gc
import numpy as np
from google.colab import drive
from operator import itemgetter
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import ChatOllama
from bert_score import score

In [3]:
# Jalankan server Ollama
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(10)
!ollama pull qwen2.5:3b

In [4]:
# ========================================================
# 2. LOAD DATA & VECTOR STORE (Memory Optimized)
# ========================================================
def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()

clear_gpu()
drive.mount('/content/drive')

with open('/content/drive/MyDrive/Capstone Project/faq_pajak_v2.1.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f"Jumlah data: {len(data)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Jumlah data: 355


In [5]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [6]:
documents = [Document(page_content=f"Question: {item['question']}\nAnswer: {item['answer']}") for item in data]
vectorstore = InMemoryVectorStore(embedding=embeddings)

batch_size = 20
print(f"Embedding process...")
for i in range(0, len(documents), batch_size):
    batch = documents[i : i + batch_size]
    vectorstore.add_documents(batch)
    if i % 100 == 0: clear_gpu()

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

Embedding process...


In [7]:
# ========================================================
# 3. DEFINISI MODEL & ALUR RAG (PROMPT KETAT)
# ========================================================

# Set temperature sangat rendah untuk meningkatkan BERTScore (mengurangi variasi)
llm = ChatOllama(model="qwen2.5:3b", temperature=0.0)

In [8]:
# Langkah 1: HyDE Template
hyde_template = "Tuliskan jawaban formal untuk pertanyaan ini agar mempermudah pencarian dokumen: {question}"
prompt_hyde = ChatPromptTemplate.from_template(hyde_template)
generate_hyde = (prompt_hyde | llm | StrOutputParser())

In [9]:
# Langkah 2: Final Answer (Prompt Ketat untuk BERTScore)
# Kami menghilangkan basa-basi agar jawaban model identik dengan dataset referensi
prompt_final = ChatPromptTemplate.from_messages([
    ("system", """Anda adalah asisten virtual resmi perpajakan. Jawablah pertanyaan hanya berdasarkan konteks yang diberikan.

ATURAN KETAT:
1. Jawablah secara langsung tanpa kalimat pembuka seperti 'Berdasarkan konteks'.
2. Gunakan terminologi yang persis sama dengan yang ada di dalam konteks.
3. Jangan menambahkan informasi di luar konteks yang disediakan.
4. Jika jawaban tidak ada, balas: 'Pertanyaan Anda diteruskan ke Human Assistant'.

KONTEKS:
{context}"""),
    ("human", "{question}")
])

In [10]:
def run_full_rag(question):
    # A. Jalankan HyDE untuk meningkatkan akurasi retrieval
    hypothetical_doc = generate_hyde.invoke({"question": question})

    # B. Retrieval berdasarkan HyDE output
    docs = retriever.invoke(hypothetical_doc)
    context_text = "\n\n".join([d.page_content for d in docs])

    # C. Generate jawaban akhir dengan Prompt Ketat
    final_answer = (prompt_final | llm | StrOutputParser()).invoke({
        "context": context_text,
        "question": question
    })

    return {
        "hyde": hypothetical_doc,
        "docs": docs,
        "answer": final_answer
    }

In [11]:
# ========================================================
# 4. UJI COBA & EVALUASI
# ========================================================
pertanyaan = "cara menggunakan coretax"
result = run_full_rag(pertanyaan)

# Cari referensi jawaban asli dari dataset untuk pertanyaan yang mirip (untuk keperluan demo)
# Idealnya Anda memetakan 'pertanyaan' ke data['answer'] yang tepat
ref_answer = next((item['answer'] for item in data if item['question'].lower() in pertanyaan.lower()), data[0]['answer'])

print("\n" + "="*50)
print(f"PERTANYAAN: {pertanyaan}")
print(f"JAWABAN MODEL: {result['answer']}")
print("="*50)

# Evaluasi BERTScore
P, R, F1 = score([result['answer']], [ref_answer], lang="id", verbose=False)

print(f"\n[EVALUASI BERTSCORE]")
print(f"Rata-rata Precision: {P.mean().item():.4f}")
print(f"Rata-rata Recall:    {R.mean().item():.4f}")
print(f"Rata-rata F1 Score:  {F1.mean().item():.4f}")


PERTANYAAN: cara menggunakan coretax
JAWABAN MODEL: Untuk menggunakan Coretax, Wajib Pajak perlu melakukan langkah-langkah berikut ini: Menginstal Coretax sesuai dengan kebutuhan dan spesifikasi komputer yang telah disediakan. Melakukan registrasi dan pengisian data diri sesuai dengan yang diminta oleh sistem. Mengisi dan mengupload dokumen yang diperlukan sesuai dengan jenis transaksi bisnis. Melakukan pengisian dan pengumpulan data transaksi, seperti faktur, pajak, dan lainnya. Melakukan verifikasi data yang telah diinput. Melakukan pengujian dan validasi data untuk memastikan keakuratan dan kevalidan data. Melakukan pengiriman data ke Kantor Pelayanan Pajak (KPP) atau Direktorat Jenderal Pajak (DJP) sesuai dengan jadwal yang telah ditentukan. Melakukan pengaturan dan penyesuaian aplikasi sesuai dengan kebutuhan dan regulasi perpajakan. Melakukan pelatihan dan pengenalan terhadap Coretax jika diperlukan.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[EVALUASI BERTSCORE]
Rata-rata Precision: 0.6474
Rata-rata Recall:    0.6144
Rata-rata F1 Score:  0.6305
